# KoMaP 2026 — 알루미늄 합금 상 분할 베이스라인 (U-Net)

현미경 이미지를 4개 상(phase)으로 픽셀 단위 분류하는 모델.

| 클래스 | 상 | 마스크 색 (RGB) | 픽셀 비율 |
|---|---|---|---|
| 0 | Al (기지) | 갈색 (153,127,76) | 58% |
| 1 | Al3Ni | 파랑 (25,76,153) | 25% |
| 2 | Primary Si (초정 Si) | 초록 (76,178,76) | 9.6% |
| 3 | Eutectic Si (공정 Si) | 흰색 (204,204,204) | 7.1% |

**전략**
- 공정 Si가 아주 가는 바늘 모양이라 이미지를 축소하지 않고 **원본 해상도에서 512×512로 잘라** 학습함
- 사진마다 밝기·대비 차이가 커서 **밝기/대비/감마 증강**을 강하게 줌
- 평가는 대회 방식 그대로 구현: 이미지마다 4클래스 IoU 평균(mIoU) → 이미지 평균
- 추론은 이미지 전체를 한 번에 넣고 뒤집기 TTA 적용

**실행 위치**: Google Colab (GPU 런타임). 로컬 Mac용은 `komap_unet_baseline.ipynb`임.

## 0. Colab 준비 (최초 1회)
1. **런타임 → 런타임 유형 변경 → GPU(T4)** 선택
2. Google Drive에 `KoMaP` 폴더를 만들고 `train.zip`, `valid.zip`, `test.zip` 3개 업로드 (압축을 푼 `train/` `valid/` `test/` 폴더도 가능)
3. 아래 셀 실행 → Drive 연결, 패키지 설치, 데이터 준비

체크포인트와 예측 결과는 **Drive의 `KoMaP/` 아래에 저장**해서 세션이 끊겨도 남게 함.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/KoMaP"   # zip 3개를 올린 폴더
DATA_DIR  = "/content/data"                   # 압축 해제 위치 (Colab 로컬 디스크라 빠름)

!pip -q install segmentation-models-pytorch albumentations
!apt-get -qq install -y fonts-nanum > /dev/null
!mkdir -p {DATA_DIR}
# zip을 올렸으면 압축 해제, 폴더째 올렸으면 Drive 아래 어디든 <split>/images 를 찾아 복사
# (셸 for문의 $z는 IPython 변수 치환과 충돌해서 파이썬으로 처리)
import os, shutil, glob, zipfile
for split in ["train", "valid", "test"]:
    dst = f"{DATA_DIR}/{split}"
    z = f"{DRIVE_DIR}/{split}.zip"
    if os.path.exists(z) and not os.path.isdir(f"{dst}/images"):
        zipfile.ZipFile(z).extractall(DATA_DIR)
    if not os.path.isdir(f"{dst}/images"):
        hits = glob.glob(f"{DRIVE_DIR}/**/{split}/images", recursive=True)
        if not hits:
            print(f"❌ {split}/images 를 못 찾음"); continue
        shutil.copytree(os.path.dirname(hits[0]), dst)
    print(f"✅ {split}: 이미지 {len(glob.glob(f'{dst}/images/*.png'))}장, 마스크 {len(glob.glob(f'{dst}/masks/*.png'))}장")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
import random, time, glob
from pathlib import Path
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import segmentation_models_pytorch as smp
import albumentations as A
from matplotlib import font_manager
font_manager.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"   # 그래프 한글 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "GPU 런타임 아님: 런타임 → 런타임 유형 변경 → GPU"
print("device:", device, "| torch", torch.__version__, "| smp", smp.__version__)

## 1. 설정
실험할 때는 주로 이 셀만 바꿈.

In [ ]:
CFG = dict(
    data_dir      = Path(DATA_DIR),   # train/ valid/ test/ 가 있는 폴더
    encoder       = "resnet34",       # 다음 후보: "efficientnet-b3", "resnet50"
    encoder_weights = "imagenet",
    crop          = 512,              # 학습 패치 크기
    crops_per_img = 4,                # 에폭당 이미지 1장에서 뽑는 패치 수
    batch_size    = 8,                # 메모리 부족하면 4로
    epochs        = 80,
    lr            = 3e-4,
    val_every     = 5,                # 몇 에폭마다 검증할지
    seed          = 42,
    ckpt_dir      = Path(DRIVE_DIR) / "checkpoints",   # Drive에 저장 → 세션 끊겨도 유지
    pred_dir      = Path(DRIVE_DIR) / "predictions",
)
CFG["ckpt_dir"].mkdir(parents=True, exist_ok=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_everything(CFG["seed"])

## 2. 데이터 불러오기
마스크(색 이미지)를 클래스 번호(0~3) 배열로 변환함. 이미지는 사실상 흑백이라 한 채널만 사용함.

In [ ]:
CLASSES = ["Al", "Al3Ni", "Primary Si", "Eutectic Si"]
COLORS  = np.array([[153,127,76], [25,76,153], [76,178,76], [204,204,204]], dtype=np.uint8)
N_CLS = len(CLASSES)

def load_gray(path):
    return np.array(Image.open(path).convert("L"))          # (H,W) uint8

def rgb_to_label(path):
    rgb = np.array(Image.open(path).convert("RGB"))
    lab = np.full(rgb.shape[:2], 255, np.uint8)
    for i, c in enumerate(COLORS):
        lab[(rgb == c).all(-1)] = i
    assert (lab != 255).all(), f"정의되지 않은 색이 있음: {path}"
    return lab

def label_to_rgb(lab):
    return COLORS[lab]

def load_split(split):
    imgs = sorted(glob.glob(str(CFG["data_dir"] / split / "images" / "*.png")))
    data = []
    for p in imgs:
        name = Path(p).name.replace("_image.png", "")
        m = CFG["data_dir"] / split / "masks" / f"{name}_mask.png"
        data.append(dict(name=name, img=load_gray(p),
                         mask=rgb_to_label(m) if m.exists() else None))
    return data

train_data = load_split("train")
valid_data = load_split("valid")
test_data  = load_split("test")
print(f"train {len(train_data)} | valid {len(valid_data)} | test {len(test_data)}")

cnt = np.bincount(np.concatenate([d["mask"].ravel() for d in train_data]), minlength=N_CLS)
for c, n in zip(CLASSES, cnt): print(f"  {c:12s} {n/cnt.sum()*100:5.1f}%")

In [ ]:
d = train_data[0]
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(d["img"], cmap="gray"); ax[0].set_title(d["name"])
ax[1].imshow(label_to_rgb(d["mask"])); ax[1].set_title("mask")
for a in ax: a.axis("off")
plt.show()

## 3. Dataset과 증강
학습: 원본에서 무작위 512×512 패치 + 회전/뒤집기/약한 확대축소 + 밝기·대비·감마. 작은 이미지(675×480)는 반사 패딩으로 512를 채움.

In [ ]:
MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

train_tf = A.Compose([
    A.Affine(scale=(0.8, 1.25), rotate=(-15, 15), border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.PadIfNeeded(CFG["crop"], CFG["crop"], border_mode=cv2.BORDER_REFLECT_101),
    A.RandomCrop(CFG["crop"], CFG["crop"]),
    A.HorizontalFlip(), A.VerticalFlip(), A.RandomRotate90(),
    A.RandomBrightnessContrast(0.3, 0.3, p=0.8),
    A.RandomGamma((70, 150), p=0.5),
    A.GaussianBlur(blur_limit=(3, 3), p=0.1),
])

def to_tensor(gray):
    x = np.repeat(gray[..., None], 3, -1).astype(np.float32) / 255.0   # 흑백 → 3채널 (ImageNet 가중치 사용)
    x = (x - MEAN) / STD
    return torch.from_numpy(x.transpose(2, 0, 1)).float()

class PatchDataset(torch.utils.data.Dataset):
    def __init__(self, data, tf, n):
        self.data, self.tf, self.n = data, tf, n
    def __len__(self):
        return len(self.data) * self.n
    def __getitem__(self, i):
        d = self.data[i % len(self.data)]
        out = self.tf(image=d["img"], mask=d["mask"])
        return to_tensor(out["image"]), torch.from_numpy(out["mask"]).long()

def make_loader(data):
    ds = PatchDataset(data, train_tf, CFG["crops_per_img"])
    # Colab(리눅스)에서는 워커 2개로 증강을 GPU 학습과 병렬 처리
    return torch.utils.data.DataLoader(ds, batch_size=CFG["batch_size"], shuffle=True,
                                       num_workers=2, pin_memory=True, persistent_workers=True,
                                       drop_last=True)

xb, yb = next(iter(make_loader(train_data)))
print(xb.shape, yb.shape, yb.unique())

## 4. 모델 · 손실함수
U-Net + ImageNet 사전학습 인코더 사용. 손실은 CrossEntropy + Dice (비율이 작은 Si 상들이 묻히지 않도록).

In [ ]:
def build_model():
    return smp.Unet(CFG["encoder"], encoder_weights=CFG["encoder_weights"],
                    in_channels=3, classes=N_CLS).to(device)

ce_loss   = nn.CrossEntropyLoss()
dice_loss = smp.losses.DiceLoss(mode="multiclass")
def criterion(logits, y):
    return ce_loss(logits, y) + dice_loss(logits, y)

## 5. 추론 · 평가 함수
- **전체 이미지 추론**: 32의 배수로 반사 패딩 → 한 번에 예측 → 패딩 제거
- **TTA**: 원본 + 좌우/상하/둘다 뒤집기 4번의 확률을 평균함
- **mIoU(대회 방식)**: 이미지마다 4클래스 IoU 평균 → 이미지 평균. 정답·예측 모두에 없는 클래스는 계산에서 제외함

In [ ]:
@torch.no_grad()
def predict_proba(model, gray, tta=True):
    model.eval()
    H, W = gray.shape
    ph, pw = (-H) % 32, (-W) % 32
    g = np.pad(gray, ((0, ph), (0, pw)), mode="reflect")
    x = to_tensor(g)[None].to(device)
    flips = [[], [3], [2], [2, 3]] if tta else [[]]
    prob = 0
    for f in flips:
        xi = torch.flip(x, f) if f else x
        p = model(xi).softmax(1)
        prob = prob + (torch.flip(p, f) if f else p)
    prob = (prob / len(flips))[0, :, :H, :W]
    return prob.cpu().numpy()                      # (C,H,W)

def iou_per_class(pred, gt):
    ious = []
    for c in range(N_CLS):
        inter = np.logical_and(pred == c, gt == c).sum()
        union = np.logical_or(pred == c, gt == c).sum()
        ious.append(inter / union if union > 0 else np.nan)
    return np.array(ious)

def evaluate(model, data, tta=True):
    per_img, per_cls = [], []
    for d in data:
        pred = predict_proba(model, d["img"], tta).argmax(0)
        ious = iou_per_class(pred, d["mask"])
        per_cls.append(ious); per_img.append(np.nanmean(ious))
    return float(np.mean(per_img)), np.nanmean(np.array(per_cls), 0), per_img

## 6. 학습
검증 mIoU가 가장 높을 때의 가중치를 `checkpoints/best.pt`로 저장함. (학습 중 검증은 속도 때문에 TTA 없이 진행)

In [ ]:
def train(train_set, val_set=None, epochs=CFG["epochs"], save_name="best.pt"):
    seed_everything(CFG["seed"])
    model = build_model()
    loader = make_loader(train_set)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=epochs * len(loader),
                                                pct_start=0.05)
    best, hist = -1, []
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); tot = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item()
        msg = f"ep {ep:3d} | loss {tot/len(loader):.4f} | {time.time()-t0:.0f}s"
        if val_set is not None and (ep % CFG["val_every"] == 0 or ep == epochs):
            miou, cls, _ = evaluate(model, val_set, tta=False)
            hist.append((ep, miou))
            msg += f" | val mIoU {miou*100:.2f}  " + " ".join(f"{c[:6]}:{v*100:.1f}" for c, v in zip(CLASSES, cls))
            if miou > best:
                best = miou; torch.save(model.state_dict(), CFG["ckpt_dir"] / save_name); msg += "  ★"
        print(msg)
    if val_set is None:
        torch.save(model.state_dict(), CFG["ckpt_dir"] / save_name)
    return model, hist

model, hist = train(train_data, valid_data)

In [ ]:
if hist:
    ep, m = zip(*hist)
    plt.plot(ep, np.array(m) * 100, marker="o"); plt.axhline(75, ls="--", c="gray", label="참고 기준 75%")
    plt.xlabel("epoch"); plt.ylabel("val mIoU (%)"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 7. 검증 결과 분석
최고 가중치를 불러와 TTA 유무를 비교하고, 가장 못 맞춘 이미지를 직접 확인함.

In [ ]:
model = build_model()
model.load_state_dict(torch.load(CFG["ckpt_dir"] / "best.pt", map_location=device))

for tta in [False, True]:
    miou, cls, per_img = evaluate(model, valid_data, tta)
    print(f"TTA={tta!s:5s} mIoU {miou*100:.2f} | " + " | ".join(f"{c} {v*100:.1f}" for c, v in zip(CLASSES, cls)))

order = np.argsort(per_img)
print("\n하위 5장:", [(valid_data[i]["name"], round(per_img[i] * 100, 1)) for i in order[:5]])

In [ ]:
i = order[0]; d = valid_data[i]
pred = predict_proba(model, d["img"]).argmax(0)
fig, ax = plt.subplots(1, 4, figsize=(22, 5))
ax[0].imshow(d["img"], cmap="gray"); ax[0].set_title(d["name"])
ax[1].imshow(label_to_rgb(d["mask"])); ax[1].set_title("정답")
ax[2].imshow(label_to_rgb(pred)); ax[2].set_title(f"예측 (mIoU {per_img[i]*100:.1f})")
ax[3].imshow(pred != d["mask"], cmap="Reds"); ax[3].set_title("틀린 픽셀")
for a in ax: a.axis("off")
plt.show()

## 8. (참고) train + valid 전체로 재학습
데이터 90장 전부로 다시 학습하는 옵션. 단, 대회 규정상 **train으로만 학습한 모델로 valid 점수를 내고 같은 모델로 test를 예측**해야 해서 제출에는 사용하지 않음.
검증셋도 없어져서 점수 확인이 불가능함.

In [ ]:
FULL_TRAIN = False   # True로 바꾸고 실행
if FULL_TRAIN:
    model, _ = train(train_data + valid_data, None, save_name="full.pt")

## 9. 테스트 예측 저장
`predictions/` 폴더에 원본과 같은 크기, 정답 마스크와 같은 색·파일명 규칙(`*_mask.png`)으로 저장함.

In [ ]:
weights = "full.pt" if FULL_TRAIN else "best.pt"
model = build_model()
model.load_state_dict(torch.load(CFG["ckpt_dir"] / weights, map_location=device))
CFG["pred_dir"].mkdir(exist_ok=True)

for d in test_data:
    pred = predict_proba(model, d["img"]).argmax(0).astype(np.uint8)
    rgba = np.concatenate([label_to_rgb(pred), np.full(pred.shape + (1,), 255, np.uint8)], -1)
    Image.fromarray(rgba, "RGBA").save(CFG["pred_dir"] / f"{d['name']}_mask.png")
print(f"{len(test_data)}장 저장 →", CFG["pred_dir"].resolve())

In [ ]:
# (선택) 예측 결과를 zip으로 내려받기
import shutil
from google.colab import files
shutil.make_archive("/content/predictions", "zip", CFG["pred_dir"])
files.download("/content/predictions.zip")

## 다음에 시도해볼 것
1. **에폭 늘리기 / crops_per_img 늘리기** — 데이터가 적어서 패치를 더 많이 보는 게 가장 쉬운 개선 방법
2. **인코더 교체** — `efficientnet-b3`, `resnet50` 등 (CFG의 `encoder`만 바꾸면 됨)
3. **모델 구조 교체** — `smp.UnetPlusPlus`, `smp.DeepLabV3Plus`
4. **여러 모델 앙상블** — 서로 다른 인코더의 확률(`predict_proba`)을 평균함 → v2 노트북에서 시도